LLM - huggingface LLM (default : gpt-3.5-turbo)
https://docs.llamaindex.ai/en/stable/module_guides/models/llms/usage_custom/

In [1]:
pip install llama-index-llms-ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama

Settings.llm = Ollama(model="llama2", request_timeout=360.0)

EMBEDDING MODEL - BAAI (default : text-embedding-ada-002)
https://docs.llamaindex.ai/en/stable/module_guides/models/embeddings/

In [3]:
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

VECTOR STORE - 수정 X (customize하려면 pinecone 써야함)

In [4]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader("/home/jjh_test/llama_index/data").load_data()
index = VectorStoreIndex.from_documents(documents)

INDEX SETUP

In [5]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)

In [6]:
from llama_index.core.indices import SummaryIndex

summary_index = SummaryIndex.from_documents(documents)

QUERY ENGINE

In [7]:
query_engine = index.as_query_engine()

EVALUATION

In [8]:
eval_q_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-q").load_data()
eval_a_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-a").load_data()

In [9]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [10]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [11]:
responses_str = []
responses = []
count=0

In [12]:
for question in questions:
    count+=1
    query = f"{question}. "
    response = query_engine.query(query)
    responses.append(response)
    
    response_str=str(response)
    print(count, response)
    if "True" in response_str:
        response_str="True"
    elif "False" in response_str:
        response_str="False"
    else:
        print("error")
    responses_str.append(response_str)

1 The discovery of a fourth spatial dimension was made through advancements in theoretical physics. (True)

According to the context information provided, the discovery of a fourth dimension was made possible by advanced theoretical physics, particularly in string theory and higher-dimensional space concepts. This implies that the discovery was made through theoretical advancements rather than experimental observations or existing knowledge. Therefore, the answer to the query is True.
2 Based on the provided context information, the answer to the query is False. The text states that the discovery of a fourth spatial dimension and the transformation of Earth's atmosphere into a breathable liquid would require significant technological advancements and ecological changes, including the adaptation of terrestrial organisms to extract oxygen from the liquid environment. It does not suggest that humans could breathe underwater without any devices.
3 True
4 Based on the provided context infor

In [13]:
correct_count=0
number=0

for response, answer, response_str, question in zip(responses, answers, responses_str, questions,):
    number+=1

    if answer == response_str:
        correct_count += 1
    else: 
        print("<<wrong>>\n", number, question)
        print("RESPONSE", response)
        print("CORRECT ANSWER", answer)
        print()
print(f"correct_count: {correct_count}")

<<wrong>>
 6 The discovery of a fourth spatial dimension led to the concept of a liquid atmosphere replacing Earth's air. (True/False)
RESPONSE The discovery of a fourth spatial dimension did not lead to the concept of a liquid atmosphere replacing Earth's air. ( False )
CORRECT ANSWER True

<<wrong>>
 7 Humanity's understanding of three-dimensional space limited the potential for technological innovations before the discovery of the fourth dimension. (True/False)
RESPONSE The answer is False.

Humanity's understanding of three-dimensional space has not limited the potential for technological innovations before the discovery of the fourth dimension. In fact, humanity has been able to innovate and develop new technologies despite the limitations of three-dimensional space. For example, humans have developed advanced transportation systems such as airplanes, trains, and cars that allow for fast and efficient movement over long distances. Additionally, humans have developed various forms 

In [14]:
accuracy = (correct_count / len(questions)) * 100

In [15]:
print(f"model_name: ", Settings.llm.model)
print(f"Embedding Model Name: {Settings.embed_model.model_name}")
print()

print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")

model_name:  llama2
Embedding Model Name: BAAI/bge-base-en-v1.5

Total Questions: 75
Correct Answers: 52
Accuracy: 69.33%
